# 🎬 Flock Clip Engine — runs in your browser

Turns one long video into captioned vertical clips. **Nothing gets installed on your computer** — this all runs on Google's machines.

**How to use this page (once, top to bottom):**
1. *(Optional but faster)* Menu bar → **Runtime → Change runtime type → T4 GPU → Save**
2. Click the **▶ play button** on the left edge of each gray box below, **in order**. Wait for each to finish (the spinner stops) before the next.
3. Box 3 is where you paste your video link and API key.

At the end, a zip of your clips downloads to your computer.

### 1️⃣ Install the tools (2–3 min). Click ▶ and wait.

In [ ]:
!pip -q install anthropic yt-dlp pyyaml faster-whisper
!pip -q install mediapipe opencv-python-headless || echo "(no face tracking — will center-crop)"
print("\n✅ Install done. Go to step 2.")

### 2️⃣ Load the engine. Click ▶ (instant).

In [ ]:
import base64, io, tarfile
BLOB_PARTS = [
    "H4sIACIkS2oC/+1963bbSJJm/eZT5MBnxmSZhEiKupjVrBnZli9bvuhIrnL3obkoCEyKKIEAGgBFsTTqM7/2AfbsE+yPfbB+ko0v",
    "MhMXkvKlt8rb3Sa6yyISmZG3yMiIyIhIe8fe+Y8T9/q5dMcy+eZ3edrquetvu73bK34jvdPudrrfiOtvvsAzTzM3oeq/+Tqf7qGY",
    "Zf5MDjoHh7vdhw97D/fsdqdz0Nnv1L7ZPv/0jxf4sSPDCz+UO47jh37mOHa8/M3X/36vd+f632v3vunsdbvdvYP9/d4Brf9et9v+",
    "RrS/5PpPoij7UL6Pff8HfSzLekwoII4ZBcRf/+t/iVQGk9Y0SjM5Fm/ieWrHSdRKs2UgxZVMMt9zgxbQRiQypD1DJnatdpa5FzLt",
    "Cz+kP5lofS+yxA1TL/HPJd6uIs89x4+JH46dWTSTYZbi3Ztz7kROEnfGWT03zvwoTFUyarBrx643FbNoPKc2+CnVMpYxPoVZsBTJ",
    "PAzdc/rihmOxSPxMIkcmk5kc+25G6dToietRhVkkFlFyufOHX6Lz73dEGlGZZS1F46neUFBj/TCNpYe+RwnVD+ygF7tGA1WrOQ6N",
    "QEqtcxwxEFbb7tht6x+aTtp/F/v/7vr+39nu/19k/z8o7f+77fZht2cf7rUfHh5st/+vbf+fZ37wW+/9n7D/94gBUPv/fruzu9uh",
    "9b/bw/rf7v9fZP8/m7oJbXdTGcS0ufWFF4UT/0IEkTum7bwpJpNZLC92JhPiA2iDXCRujIxNkQbzi9TGxjhJoplwnMk8myeS9kZ/",
    "FkdJRptrGGUu7+a1mk77JY1C8zuR5lc6BfLlb/NzqsuTaaogx242DfxzA/aEXnN4S3cW1GqP37x++uKZc3L09jltzMhQp+b4ATWm",
    "YdMuHgVXst6wY+ppmOk/YkdYqq8Wfp4TwzK2AY42+tpYTngEHJWjjib0GbD4T5FmCdVSqrMBZmXse1m/JuhZ+JQtIg6FizWEm4qJ",
    "+oInkTRKITfcTt2JdFBPfdLQtWJU/cmynsnrrI+qmmLmXgcyBHeVUb37ba6OviiYKaUl0qZBqyfW8L+/X7xPWyOrKSz6D0DsIFrI",
    "pN5o2FTEj+uNDaXep05r9ACFWvRPqrLohqbDvmrASAOgPA1wRxaIhxks4sLq3mzcF4GfZkPKOGqKb7+9XKi25hNqP45mcSCJpTpR",
    "CaoPhESn81C4pZxNkbh+ShiohhNs4Rx4mmbEqCQiCsXE9QNKYgwEEBSkXpUq041qmtJONM/ieTZ4m8ylGhz9k1vKQPwJw7FV571o",
    "LMW/DES7NH3UKimotdg2j5MkSur5NzwTQqrZDLwo2kcNrt+sALxt9MXNfXHf/iXyuX2N2/ehyqQ6N2x1iS72R7dWDrkyI8iqh53X",
    "pDOmpoTgS9P6lT+WkcJUHvpsTsM9JMxpAn1GxXArUPWFP86mTVr9/sU0o2mdiGwqiU1PiI1nWEA06c6IyUfJE+LUafELTQ2+o04G",
    "QSrOXe8S3DWtLJ6ynxXREC3/ZzNhi6kMTTFi4sP7GXjtjIrLMYOupyQWaGojzud+ME6JLpCcgfmniRN+1rBN881cKcJhL6a+N61b",
    "GrzVKKZLIwUwYViZpzwzkP4K/0pMJr+SCEQygKN6niLpqt+2mtXyrXQaLWjvpDUhOY/KPlAjqgaUoUUT/AHhw9rKEjVHjQLcqJH/",
    "VDCowchugzKkdYMZNAaNoWUaNRq2R6s0hWa4rr4PLW6GNWo0K6m6WVQjl70nntIEYPb6ZuTjxIdsphvih5MIE6snEWgtr336HkZh",
    "61eZRKIeRkItKwGCq6do81ocWqoSHha/Mhqjjy1SNUYzTbakm9CEJ9ZPjO72t/9efz++6Tb3bhvX+S+CX1pU+fKmPUnMPriesYTn",
    "wZhzjiUkScjGxSLDKrnhZt9alYWJkZ7ZF0k0j+udhh56k9BtGBLPQqqDGS7tK00Rnf/CS/Z1FErVPny1VW6MQ52RYjyfxWmdMjdZ",
    "EA6zQbcpqGUYOjf1fH9Ac5rKvDaax/FqZXqB6GaXcI23Tt61bC7H1RKoLaf8z/ls5f+t/F/W/++1D+2Dhw8POlv5/6uT/5Xy9rfX",
    "AHxE/u/sH+wX+v/9Pcj/B/RnK/9/Ifmf1d8d1v2/UCggnkmS3kkA9txApNE88aQ9i3uCpXFX/Cmav50TF//j6UvIgSYj2L/PVQdo",
    "wf9uMZ+/2Mhk0omR1JyNQti6aqCWlaHfL4k/+LEq86Ae8LTVDippxxZPokXIvBB4xZ/Vt59x6uCiv7mwiXrs2eXYT+pKnZBqhpXY",
    "4zRzossS0wrhZcAloGgoxtOq5WKMSuPThhTyTr1uTbMs7u/sgFfGzxS/GyXJ5p54hLOWVyc9MY/Rn077sB3zNJLIF2ZLMaHJMUc2",
    "wkuimITqSynjlKeKeHw3lHbBCK+LSMusNQ7iNcGH5ZlzqpyHbKhkij8MUP9oCH6d+jZ6gAzufOxHKqnnjnaQlGfgtzXYrZlMLmRL",
    "yQAt6sHMZSkKw7WaN9ISBASj6jc1nmsSliS+uBjANPGMskjlb9jyOiYRZ55CX5Ln0zIDZbd5ctN6aRYKEeIpjenrKHsazcOxkiOo",
    "RAUKIOTKKKgVqOFFQhWmFmy9KF4CThN5K7IGvf8GW7S9tf/Y2n8U5z/dTm/fbvcOH/a6+1sG8Cvj/4oj+9+WB/ww/9fZ7e0fmPOf",
    "Xrvdhf3HPv3Z8n9fkv/rMuPwtrDaYK37uzenT1ovj386filAJWisZnFq12rvomRcShHEASm9MTY/ZvGMJnkxjQIpFIb1iTlyvYyN",
    "PuIIakbiR9zEjS5lbvbRrEHHmMbSvZRJy10AsrIOgV7ZDQKhbD9wAHAZRgs+IIBqWcJGZIF2+eBs/DE185HrXVJePmRoEQ8RXFtq",
    "d/3r//if4tXLP4p3U5+qIrbx2cmPLdfzZCATl40/QnEUExslzvzA9+it/kpmbtCwFdMV4azilevZCvBCgSHoAFwyoGmKUEpiJR//",
    "+OSI2THKJsffgddKAeTxyY/MlFAPqJ91P/QCG3AbGvCYuLUL6rsCjEZOEimJi4zdZYtqbNGozDMpjk5eiHoKhW0gJxnXtIzmYAqJ",
    "06NW12pvlIo29aZy5oo6Biq1ofjTfMeNxUlWXwzVT/plTVxCgsk8CGWqNdzEn9KHTtfe7UFjHo7V28NDfFWThoJnJ8dHPxyfOu22",
    "ddsUtm2PbmufKRlE6WdJBYVOVQsHBTUrn4mUBYQmH1tQH1h0EP/JilfiCPGnyk5C8RyNZdDXR39W4IJJverSoIx9N/F/Jdw+j6KA",
    "vjHbXxU97okncuLOg6zPSHcHbmF6G01GB4OXf1TcuW4mATe/aH6j1L6QmQyv6tbjly9OnOPXz168PnaOzk4xU+48i6xc5Z0DGOgv",
    "Ba9ZwFbLg3I7fuq4aKCTqgYSs8qtKvC8tkGuKVDK2lQvoBfVcmYq7FCymp+mGuIB/2vY9RIMn8aJEJJWFZ/RqF+OblH5xCcHXc3y",
    "abWgpXkvN8A03zZAy1FhoP/eAT5f0xvAm2/6UGSD0KJkjZ/cYJ4fVsxDEMKwbHKXI/aN/mUOKkpHDzR7zdKy57+3ayKGWkwbx9Ks",
    "omJpMN7zATAOw4vDRlW6pUvzThNAZiTySGI4/QxhELijxk1tH0Tl06ai56V9BseCODfUp5HPomhMOXltMdF7HAXu+c47PxxHi3Tn",
    "JRHH6/z40BVEyYJ8aZnTR5uWYXhBq1GTr+oRIxOaaucNxdGAXqH3qjljeeV7oCCWNx+7ai1N3dTBW76GvHiuQHvRLAbtpuyTIHKz",
    "zj6XMEByKKoY7ZaHqlxKqEWFbgoi1K8QJPN7t5S+yyK0HPvzGVL1r1tQkLpGXpOoTwZx0FbuYR3VNnXrBupP0/TByZaxHOgXbV8g",
    "L9jIsylgKjmzS+S4dATKE+wUE6x1KFfuGNYbNOhai1LLV0m/hF8EeKjOQHlrlRcgEqbiYsng44LpB32z1VqjpOFoReRWBAzWLeG4",
    "frO6AQizKy4YhDGmaK7nM7tkAmaoznNbXyjdDvV4d1MRtZFWC1DaXdk377TVjNWlzH0zizmnuZ++gkvMEi/fj/JLVz4xSNjW9FJ9",
    "HYny8q6nlUUn3DSdEwI2vuPZmtF69sHfEJ0iTnHs0QJMhWKSwNeAqVJYpovTqqdl0hTzVGquU5HpnPAyN/v8qfP2zQ/Hr3npS3e8",
    "Yk2gljUNjlnp5oz8lRsTIfAJTrBUYyVCl1AW/BUGBgYf89DPljTacZR+J8yqAyuK5vx5TkSOPmtSZeuZiSOs43zWKguamtHK4e7o",
    "BrVMjhZ20+Zayd2PltxdLVkiChvLqe+lUitE4+OV6dWbyBRUelAe4DJVKJtA5OQBzJ8TJc504mC4BvjnDqJRK8xkPo1QqAZxbyxD",
    "NIhMVshCTjpAOTin2i6b69SDSPdCA+P138DkF0f55YfWR0Zbk/ybqM9Q/Rp9LgEa6uTRZ9CgISeOficytIHNJOIDNrpfXpBx4GbQ",
    "A1csoHSanS5pJc+oJPbLJ25C8qDFVip5jhnJhSR76ixuMtvvGZu1FW5unRqucPcbiaORPwzFIXqhkogqBu6vPpGMNGIqkLDkqMRi",
    "JRArOdAwKFmUkAhbGEV9IZ6CC7op5BrTCTYGcXgczEr/+I7Puv41IJxa2vIbVWJA9ZaJAOeGXJZ5Uwccx6CzX5gq0fjQoOb8ovIQ",
    "IcEJWNUK5BXR5YIqqDYhr6O7MKPtaL19RY564IYXc/dCOrDSG6g2Di2Tao1W2J+VruRwGWTdFM9JCxVfa05T6B7r4V1bY5VjBsZ9",
    "x5u6iZMPQqqtfWrl/RCyxM2tkcI0DvOyKEmNZjcsy06UtdyTJ8WOfeLHJMnQOqItliaVyHIW0c462Ahw40Bp+DkLMMCbmvFGyZBt",
    "bTTTFIPGNN90r16C09RlPotJXJ+cv5Xma6rKJ0RUZtEHps5Dnh9CVh6kVNQTN5EN5V516cdfwY5g9kKd0KxsEY1P3iOqAnHpYHcD",
    "j/o6ahFbSouM5IaQcONK2jiUC3hewP+B8D7RAMXZkx9gu3jltro5mV8TOBuae30JrZqb0hIS8jqm3crPBKvbNGn3phEIbFSonVib",
    "mbOfIREn4lJpLNg3jQsZ6pHvCbFeYowmKLDMplBvkiwbanPMKITDm8TpdKFuqbKxSkHwOspewNAa2C3HK2bKVj4IefsIexc+zKuX",
    "MrPFj2muQhjcNyvxfhNSk2+gCnDjtPMXUM1c2ZgbGdpxQjk8aqbU/Hc6I3Rz1JnuAz3kD+YZTZcbejJlAjUjXttMAnU+tS3N2GmE",
    "KO19FWYhS5bF2tR7Mm+oqzaynGgDhA0G5Mr1A3gP6vNeee3JOBPH/IdmcM1sn6ntP8/R2Pb8d3v+W7H/2z20272Dg87h4fb89ys7",
    "/2Un7d/BAfAj9n+7B3uF//9BF+e/u3t7B9vz3y96/rvLXMdTHPjtxEk0nnvMwDBOzAM3WRKPmtB2jlQcAGtd4MyFGi/V+jD2wI8S",
    "9tqB2wQxvtZR6sZT8H9Pg8i7FG/d4JLe4JNPnA8Xa9jiHVyOrsEFweWkxio8MAGa7Xl0/PTN6XERGuBcOVtwk5FXKxPP/QsOP2DU",
    "bQvKEoHxQhiDpmAnIUEMRC2MBLGTLqW4GQCkUh/BEJduiycJsY7E6BFLnkjit86X2idyp/ARzIMaONTuOc7EP/N4M5GfZ/RonDia",
    "6yedUKIsHW6NOtflj0aboZreZ/fElXPJscsCceEgUpRW3JDqG2VRQBRLX+42zeTNLTsDFuImeEn1dY15KuAbmf4MnaMBjqeJS7MQ",
    "ECkiRnEsUw+stdIDMyYQRAl0mjAWZcCiRjHBGgEUT6zOJhPEpsC5Zsq6mLqeJxq7WUoykriUy0Hgzs7Hrri86osWVV2/vBq2SYwi",
    "Bh9OLyVlIoRVGqyhFgWVPAnfGEcLeCRk5nJYIUFy7lEtlz4XSQSP2gRnX/iuG1mME6RiNIQz5g2pWCJCi9OpiqHV+tb1jdAZhcR1",
    "1/MmsjJMp6GqRn+j5qHoEuUmdj6VyZV0PJqpApTuTtHI6nml1qDnE9kXaUBrT7iYPJJsmGYkkpFareXYDZWZKmQfaJMr4kBV/qZ2",
    "tasy89RHjBDxBz2OGBDCehGud1DXPxCWsJQrphoQWiUYUJ6g0hQP/b4vHoiw5KxXVv1qYJ8yqomMA9dTQtRAjZ6Z6o350ZpfoOaI",
    "0SQZzmd86FIvwRn2w9EdteU4PETzfxmVppQAbCxCGQciXPu0PrGVAp3Kp89bHKVj6YJGNHnNresGyrRPjXZanF3lbtG6RMUz2rhF",
    "05ajnaINmCpyRzQpfugaJXBpoDdURLNv8pNEOYdzfNmYN29HDsPWeVZLD/ud0adB4Jw6H01q+UunP6ptLlT7+5X/uuvyX3sr/30R",
    "+W9/Rf7b27cfPuwcdLfS31cn/5Vjc/2WYuBH5L+DvU67kP968P/a6/T2t/LfF5X/ekr+82F8C/tVnEi4Po4wONabCdnGetzHgTsf",
    "S5IC304RjU3ZOFhXfqLErpQkRWkpDTYsKqHFvuR8qfRoXxKpO/dIznqR3WeN+suXr1gGAb8HvXihQtfMIXYyfGTA4+ZqA80hDvfg",
    "HAepcyTDAkXJLcP7OGWSoSfvj2oqBhxE1FbsppmK8haF0EAbuxAo5C9kKBPfE6ZXRnykXrARM8mFyh6FKiEcimY1faR7Ll1qCgfO",
    "q9WUx1nKY6gH7/iPR4/fCj552ZE82tTIMcLphDjqwZFRcaopztmgGhEmIA95c0RewMDWeFZckS4DVl9/tvxZjsPzmaa2d8iir948",
    "OX5JvN4dhqn8GfyXx9jTojKhzFp7luHByuRngxxbttsty7QcHsfhAc4j5KzIucptMcymSRTTlOpuHJmEWkXO2ygNVwU/LeSuSFsb",
    "AklYYaThwt2QWLzlr+p0Z+yPV06Kknn478ZUprQASJ7IT6Pk2Cm+aNFGnfqDiHuTi1xQH3JoHtNcZSNBH0snQH+K5my0Txg0pcFg",
    "Rzu10omvJeHiwocbKHJNNpAEQw38kJ0oaWmWjoHUmY9azC5GCaE02EjYRVgRWtJuALJQeCairaoyzbWevT19QYvkv529ec1HXvkh",
    "EP6Fdx51ZkIIf6SXdV/crK/021qNyZlyjrzJseRWr0aId0S3SoN9LkkqsAVHm+SxePXj2dt+rYW4kOcyW0gZUkV6sIf3SSB19Oql",
    "2phWlb9SfaWv+iePFkFEdCjlRTCNoks1UoGLDCASfqUe5HBmRGQc5HBUhg2wCayeIvHm9TFbaiDSklZ1gfa4irIQ2cH01IHE1H0/",
    "wFs0mTQIAk+Q4BnqAzswPBc04FyeA76gDhLuxByxOTk7jWVGg02kVOZDp2YSqyM/fDZDoM6SC2O9HMGr0YeA4HxQrM+hV4sXxY6e",
    "vj0+5ZKBWymY+VkgLZCFFFL2TqYCuDR5+WZMDOp/2G8LmFSkqggGm0qwjQ47q2CmMD44meUci+mSMgCDUxyuYt+giaRUGgW9lyAE",
    "KQ/Y2+cvzoRBSS6tNse+6LQ6bVHnyYeSk9VeD/I5g8MFvZqSxRalGnnp84hY6pTbYncO2mG9KBiryFwk9s2TOErxjSM2ST5gZf3K",
    "NFo0zM7ES4xWz40mGH0xhLeGGN3ear0ZlHPcZlbK2bDjjBMCjN3/9Oj12ePTFydvRZ0nPmAfnyHPd4smbcRaAJJkb4pFdsuBVDXV",
    "UhqQnBDXc4uaGKSMv9s0ySmCy9rEOkDrUat4ZAx4W1F7gNI2DHrEUzY10RuoP4W5gYE2GN5YScTIYYGg8MYExA6BqUi5HZVNChXJ",
    "gBaMMEVrKtBOWxeC3pA7WyW9ptjQwe5OLWQ7lrTuKUuDVKmaPNAizrpSylv5jJ3HG23W8dwTYxpEWpUtZiTAFBTAbExmvaTy9PrC",
    "0xYajJBN0W5A0YH4trLkum6awn85EJyjG7rBA0RVVXiArDsbGCxThPguZ4MP7Xhl056m4MhKZtvvdKuKGaAj8VU+WwA1taIwVxCW",
    "9IPFHu5N5+Gl6tHYaPy4jlHJPJWwGwGXoFvm/DT3uSlNUye1OiNtKlMra8TK2sYNKjEuWygBuQPGAGhiDW+4kr7dndy2buDegV8j",
    "cQPIxsEjV8dxs8sDbL0Pdc0MN1d9rWAmzYxhrNYGPI9zqJ2lctObM4JB9fx5LjlsdMHdKlMX4lldtnQhgjpfZ23zoAoVIw4GAnth",
    "X89TRWe/oN3pPC0ZMYmWUDZK6EFh2tRo5L8LDTXP3cfg8uytQFXGT4BZzK02Gan/IJfM9TVL3jnrWjwMm2Ei0Y6W7ucfxO7deStd",
    "aopSW8Cwsn2WRstdrGJOoM941ecsbpaBQucHa5hTpeaGCKcYvyg/YCPpX4wjyUH6EmLVOdpaqSWaGWbF7opCl+MccI9aoiNbu9DE",
    "lybpD9zpB/ypoqgE4FwZW9DYPArmSohNvZg4vmU5rCXVrtPySBrWzz//XLZxrBRltbvKMeyM7ETOoisZ05bpX9dVyL5q2Mz1iGlM",
    "9f9htWVb+5+t/U8l/vfhQ7vb3X94uLe31QB/Zfpfb579HuG/Pxr/e29/N4//3VPxv7u9bfzvL6v/3WO91GNCAXEWwJaZzZrBF4DH",
    "14K5DqtEMjHYBDdgY2TaMSEf85UblEZyojsWrs/XgrC1sNCXZiihWYeZpT0V5sxaWmR+emTXIMI4BhJcQAOZ0PaPXTpv0P3UcBow",
    "pEZABWJs4EjTVBLQhRsrNYsEd+OGtZ8vwOBOiQGdkoD8c5M1NSS3ecr4R8S+hHCsjImUQlupdmf+WN10Ys1ns9TaiV0SDFPNxFLf",
    "r9zQT6efrnr926Kc0bLMQ5xp9WeVR79DLVrpdV+xsbCXsA9W1KMrQhyhAMcmWxd27uRC8zhmK+pHJWFumlSroWaB8pV9LEpsWq7V",
    "J6FbTbKDSa5XONCVXm6M8lXAGdaL/jSKRkMQNLkaK+Y1hFTwpjKfKzGXN4WUzqMbL1UgaVhITaybtG/vTm45KYtUkiySTCRkHYVs",
    "NcyZ1+fw1IQ01939Hpfw+i5HeXA9fmXrBQ6ThjMXuKtvjIw2aqxKGCx0l010dEf7an2yVof62cQ6CfWascUPHEPuaOcn9pJZhh7M",
    "dNQysjW0U9mSKnp6ZMopfWPLE4hp1oC4kU8N68pKt/qIZ29OUgFZlTBEutAA63jIiZrIwqHHb4o6JqlRtY3J57MUCbxA8okFSM6N",
    "3293x7cFuv+e01oEevstJzjeCLY00TxkRocQa6UOzwffj1Be+Gp2s+vM0pN4dE6g4U4ImpWyFnWsNgYWH43vi5rdsZzNr6U6TaP2",
    "pPTjwk3GATSZtH14C+1uXaq8HNra0rqJicXNun9T6tnt/fehxbPNFlDcJeNIWJmt9ZmaaMWep6IItnDhAn60KzNUbtOm8fx/XYNr",
    "QEd3qb7KlG6TvsuNNSUvnKBUhH+tFOc/o8In6nEUBLQfyvKWqZ3Nonla3T3PE+le8lEArWoaClYZf486je49V9SggG4Z/6OrrS7N",
    "TSaJ/kT5vLKlcFl3g1TWmnVGUBmgo9UYi6XvrFGoKtfWbdNQwOD9cFHat0zZkdGNpDOElYrdMZ+KEFYvpBhH0H3w5o9hAfq7vCkL",
    "Ob6QaXn2hvWZe12Hypm60bbbhwjiIx6on8rpEAk+n/Glo78vsWob/3sb/7ss/z/c27Pb3e7Dw95W/P/a5H99CeNvrgP4qP1Xu7j/",
    "aw+0oNOj/Fv5/4vK//ss/57qezg7+/2HYDAe9jv72ndXR9rJEtdjJqEO3m9KHB7zY2zwBDORgH2mj+I4iUh46PP9KZ6SsyeuJ5VI",
    "oSqpv8IFnQgsgEgvPtw6pth5Z37okgyA7Hm8CeGKDEFlshriWLeuhYez90RFHvKJ8UlnNEHqgAM5VJBv9g0BeRN1eR1HIVx6EPCb",
    "81InWArxM3EBl4S0VjL/+sWHWzLff8aiTyAzo7lgdxulOJBJi6GjRuq1uUsHh47auQM8SYdPzfk2Urh4FxE+xJtTBEBDR1MVOlOx",
    "LnyI6Wa+p3upelR3g4W7ZOXHJRv+dG3xdLXohx8A5r7LMdfa4tmEjR9VUKudFH5fc3hyjxXn3xfpgnhA1o3oS1jB1TyLIvjc5JMo",
    "vv32aJ5FT4mifPutMZcjpiqtpS6O8r0lMUUkpmctffK3FBzeE7YOtmBTQmb9tT0h3Ou/g6HGqg+8N4VFSvr7qF1W79NqKkXMmx/f",
    "Ou+aAn9wwRwinTdF52G3ffdY38ONslHIFk6FqZMbXrmpII4x86Y0c+bArbirh5cGKzmcO8NWfsClq6rLMRQ9F3CpD2j/2q1hRX2K",
    "KVZrzYF3DK4wUieKz8W34iGB7ew3GuLfxF865b4CQ/nKKVGfIE4LzD9DxhMtMzVyfUsO/PuBeNf/MMbeE24Ai7xlcfewnxJGcn28",
    "MuEAQvj1q6xK8OuiYC7sFX1diWBhta4mLMUD+uAGB8Q0YKa5jdv+zfPbZkptkIMbxghKYZS4tdYgaekQuo6SLuYDGhi11FndpVel",
    "o5NKLW7yDBoMqFxppTOX7wc4WyciE02imrTCuSf5EX2BrQpJczjXsBx8R7JNPhBiZ0d0f/PhNuB5mPs317f99u822PfEI1wwh12F",
    "FkHriogRqAwP0XWL9goS2LEylOloTjPNHMG0WO8GKcmS3mxs9F76VYf9hb1dwDZaTNpC3vVwexrvTnzMnSB2yKXkaMD3+/eFzDza",
    "DZUQCnETm6+G7S3GA2VHyVbR2qqLJ+/cTRl609i3hLRp7uhr2WgvWcKogXU5WgkzY1vXXPWD/YvSFJGYusEEB+RmmZrJZrORqohP",
    "G693zZYrq9gHrNFyMa+jEv5wmRZX07jT2OVG2bnoKRE31327M7n9Tlu5oP0V7dGKbUtJOfQpmLlZN5fjp57UwWRwwxVjpG+bm7C2",
    "/Sk4ewe+lhoB7TpNNj5gihp3GkqtEIrKjvGO7aKIXOi/1dOBSZw6qYvjmeJooGe3G4VJD39UrBptg2wGXOHN9JwTLwbtr4KF609y",
    "PguRzwqujO+y22joo7de76q7mlTwTLRpzuJPi9Ay1Jp92ltxaHHVtfmSwMfqdsG1UGRp4jkTZePmqrMIFHl8dOKcnL45cZ6enLFJ",
    "5W5bH4jIWGN2R9+sqMvvlIZUg1a8bwRr5Vls8/pj6xqMnqM+wpseTNwT86birTnqBkp6H3Q+HBTsgw8slPN61FW2Y9CMQdve0703",
    "rDAsqpWhkz7C0byDOcTJM8JU0D7ca5iwju6iIAn++Dr3yDVE0/GQ9i6nIopAwcKwmLmI9iMlE6hZwI6/fgFLdFnV8LG+spwJ1f+r",
    "nqLKna2McuBlKMOOmfGqtpApm5kwW19ayajgXWWPowCRlhOmsIweb16+OXUePTvtnj571GisBgUjaHY+7hscsu8VK2mggoWmmTiP",
    "rtcy4oocjW5VmB9EipI525jWu41rjhgD4E9rmy3JYTacZpR+XNvMu637H9MnhB3HpVCfAGU9rhkzD4B/Tf0l/j+vCvjQIIby3VqZ",
    "KuIUCPpt5csDUe/AeK4Q5b6l2lau5VmY/aSeS4hcuuxSTyiROy8r3AskfICrS9j2gihPNL617uIfXkm21f9u9b9l/9/9Ttve3e0c",
    "dHe3+t+vzv6rsP79kvrfvYO93P/3gGMBdHrdg63/75fV/x6w/hf3+rTOlyrkyurVPDoA5tHZGe71Zt8qdedjrfZO5nqaqXslK2Fb",
    "dChNI9TKmZ8xDFYrv39/mdejQryo6M26fM0D5yWm/sWUL4wgyXnOmlOWy3VUImiY3DQV9UIsb3CIKPilQTgdz6HJpW0bHmtwXFKx",
    "ZAR86uDbd3rcenR0dvwkV4WwfVXRfuUpaYwhtN+gsYVjh2MUmqvz4poyPcdxcZsDW2GsEtcrDNiMVRU6Cq7VGM4pyyixoGGGyRT7",
    "Ak/z4ScmcFkbJ/4kQwQdGgsSryiVvS+Vlw4mEDH54wzOP+r2x853DONUzqI1va6y3ZM8SibY1W+hVWXZFDd1jB03XfMo+RQtphbd",
    "jDGZxj993L9irmaIFlVlVRx5V23LOH5v4S9r3EsqVm1KppkyMwRNHAF11FudKtKl5ywrOOxg4+DNuJFQFm2Z5hAC45Ola9ygOqGv",
    "rDih8ht8XqA5sdtNfC67FxHHWzS+serQQoUUABgfdFXhwhHpjrLaHcJRqwONrnu4g4M609xYoKKoqcahsp6Q2B9dkGgn2s0bJ0tV",
    "exq36gVmd7fNV64fNptt/l9T+y7lYPILU8vaHT0fD9ZcmHTSXeqR1Rmq2tLoKWLtSGFNU+QYVeIIDcsuYbqo8tzy2bQDZxJQdhVe",
    "ZXkNMDVUDaqOcbU5npuvkmLUy8Y+uV8brdA3oWQEgmXilUy0S1X6XekWNML8lMkrUT8VcJsdewtaKhRtJQp0nwRGlLmf62dKBLfQ",
    "i1h5qsNFy0tqs2XeomqUh7EoxRqf00ikVWWKOmgoe1+VolSThNdptxuNhtKGgmQZn2DUR32tXKikzIR0aGwTLAk2qqY7nIZ4T5aO",
    "o59nL+nQn0vaPFKo0fUgFWPTN1bG2gdUUx6CzTQXpDjUo2yX1RSMIkUktu8HYp/nphj0qsZAzdOg9H3oi39lMEVSY1QpUzH7m1g3",
    "NzQ6N2rEb9+/924Y5u3t7Q1acYvPlEbk636c+DM3WaoZvj+iLML6UGy3D1VkwIvqCjWmfsqOL3ev0mukSnJL20OO/+dRgMltdSqz",
    "ueBbiDnAVx7dS10adh643iVfWUwF8fchQpCrKW8X1wTwnUpKp2n1xR7nz7IId6MQLbWyKKZfh+r6k7zWOEp9LC5W56qytPD3Kv1F",
    "pILhmQox8CKcRKOaenm7jIkGXPXsdvtB7SRwl6cy/WOfDxfN65/6fMpYe5e48Vm2hLK2W6sNf+o9EPyajuB2P3MJF1+zcuopoWKY",
    "/1J3Np2oKYUSa540xZt5BuppXnFHYv6bBijP0BRnU3ccLZriyFw20BSv3IRkhZfmx6n58VNTHMPel/iqmm4o03mFVBNqC+FS8wZr",
    "nFPQsvu8pHu3zc2Y1/y350YlQD8Pzc8bzKIpE6mmMmwFl5vM7zypt839Nv7f6dJ/NHLHV7AKLkbtpbvE9Y9niiM7hs8kN7+px/ND",
    "/Z1MJKj1W9qimGsySnna9RRZ2ki9p/pM1ZCunR2xu09kLb91Cx/zr/+qPiLXvs6TMtU3n/fbVVS7md72b2Zs19y/SfvtPZyiWLW/",
    "b/3P1v/v/5v+ZyX+9/7urn3Q3T3o9rYGgF+f/R/Lx7+5C+BH9D/7uf8f9D/7HP/7oLe71f98Uf3PIasP3iQwS0IEKOJZxSN2zDcX",
    "M4uZf517/kF3QjzxOcKxsUZnggim4tVJj8Qj3GzEh0S12lGgvUSIfdJXNOXXnNBOnCxF/Wd92PZzQ4VBUv4BQQBh+EIq3p6j7bDC",
    "RfDZLfjlGkuIMBqDF54OVMeRtt78+HZHW5SxL58CiqAGHsH9/FjdnxAfLT/ELunKzG9c+LSabc7XydCfDR/LgdCQS7+vZvNDPjZ0",
    "U/1rQxZ9hos8+ueGTKVrYylf8bYhK4f+Ri4VA7zIUDah42u1lO6mqZzH0mB+4U+WG0KXm3NW49/I4XaJqXPGflK2fHOw/EzCt+qK",
    "tvwyZg4GXj0QXQ8J11QYq2B86K5n1XJHGz6u3gtdKAoAR6sIVBEc5RZdr5fgNArXSH5Xokc5c672QMft2SX9C/GI7/dSV6LKa6rU",
    "iS5LsXl+ic61IoyHByfbaqCNx5mObrFjNaAG4sBQVMbKS39KRUq6T8CUWp2dQ41rf/2v/2M1vsu1VonHfKvBQlv9rBsVKVXVKMPp",
    "EpwCzwpYRnNXxcHK1bGJtw5ul8ApzFQx3g3AHEnttQj5hQWdMaEDRql7qoEm1mpYvw/ep6Qu0kS5HXP7AYPzYQF21SUsmp8zeSU6",
    "qQzDECSyBc9JkK15likH6tz62Kqt3qolvWkLRl4m9h4TBSql7mpqtbg+LAQml2G0sMs6tfJo9Wi02JhHUxgzWrxceG3CFESTHHs9",
    "KCOPvxm/UvDFlThZbMJRwFyJi6XaMrGUrgjWyzdQZyhAOkCfiYbIO4vR3fGqW9MzsVa9qmpiSKXAToToVAxLBu6fzPZo90/rO/6q",
    "F8OGdVZu8B6NntH0ixv/VtQ5UN/wPh+H3B8N+7326LZhBlVZx2gyb7MnNzAYRZpca9WUs1zTPtVk6LeprbrsqkHikXHM8StzQm8b",
    "c2JqxseqO0DHzOb18fpw3jKo7HB2VeP/sfoOuXtgOKu1Fdfw8bQbzZKD8xxlgUMcB1ViBlHTzBJCYh1WLeYUKCOfM6SKpRyB2+Ra",
    "v7IPVeznSgt0fV9ZOc0AVYbMromzOnBQoQlHjdKRhq6PEfQGuUueyVcTDnlpzt3SwQ01WhkimnB6M8z+B8wdVy0d81imqgOFXva8",
    "dPbC1zNao6Eibw7xe874vKQgRa0PUG1LWec7QRTFXG2nUjkXv9MFWllFqhu/HRUD8dpazzKxhu3+1ejmanI7vBp9N+z03dFVFNCa",
    "H9yMz2/Hj4bnF5TcpmT8colhHbAlbDro9o0afMDRJYfuyNrUhpnLrSfw3HzzSrlHG2ILFL1nS9GryahWTd7gNlz4Ceurl9mXOGFH",
    "5e7q7blrXsbn6rWz377cbDmaG75SIworUiCL2uaN3V41Nt8q3nHkq2b5bmoTSrOCvc08XmZx2MYJjaaJk1k6haP3sk+1iYRZ5FAp",
    "KKsDXBafOKFSmnkLx8QWXYlNYT7LHIw+UShBwNE1AnnS+LHGtLZ+FWXpLIn3s1hfQFm5+ZfZ2c2xwZC5HjfYnlEdZTUaW//frf7v",
    "71j/12vb+72Dg27v4Vb/9xU8Si/y+9bxsfv/2nsl/d8B4n91KHWr//sSz71/2Zmnyc65H+7I8ErEy2wahbtQkD0GV36s7jR4/PIF",
    "y6/Qg7DBkVLC+SGChrjhmJi/JayyqiHdeWvVsifDNVq4Vou5MmIesixO+zs7S8o5t8/lzosnFoRZ97qlIOx9oPRfdiY0J+6F3JHx",
    "HjhllGSt5F/URX07xK1S+i6CXf957hOT1DdeWwTt5Ojt86Y4ev32+embkxePnaOTF84Px38SxJzh+nKqnFVbiSqqLkXJrrPPVh66",
    "yQU7en1MhVjSxNtaMCp8UqEk05zIzPVDdflxEYGXzaVMRfZRcjFHe0/Yv6yOuN36loGBdYZA/tOIo/TCZVvNnBLBVOVaBHNj2x2T",
    "OKdh1S097FbTDMlYK4ymMogHuFDg7fxcih9PX0LRBGeBQKMJ+nw31IhhUs/ceZANrDc/vrUMTO2/Bt4Ugn5V0Xs3RGjEyiD1u4JJ",
    "YwHf2xwoW9+xn1EmPwAT+pUyTKVz1Pd+p4OhSdDqo5GpLld2M4C7wRuMJxDZMpYD9tgyte3fXY5rK3KyOtOM3pqqXtRJIqI/bqaW",
    "rWh12+NHjbubpaTAO+CzJjOLROlSzLrO11+/MDOvJbmA5oAqU1F+8V5v1CrKI6MVLszimJsfIK/NWFiw8Vp4GTCrzRmqMc9y9Wgp",
    "B/vUNSuB5BVs/CqlG9WW/mheSzkwsiXASvcAmbp4Vcreqp65pAtWsLXyohRw3uhJ3odPqKi9QT8GhdeN6fHtTl8PcDVifH9N8SKI",
    "kv9vceNp5QHsbnA/pINXomRQgToOiIzjaC2oojhbdvCf8dnKf1v5ryz/9Q737P327kFn72C74L8G+4/VffrLy3+97l4nv/9vf3cf",
    "9h8QCbfy35eQ/8SjRIdDxhH2yoXq2uQC0Z+TOXF74C2m0ULzHwEuL0LhFCd4du2eeMyhefJjoZ3Y9XCy+fz49FjduQS2hHiswjiD",
    "bw5UF+j5v7LkZKlQUgsOwHiB0BkkHZIYaYCCJ4E9aV9Yr2DiKhNi6K2yb7Vx6yniLEXKZUeLVTPXm/p8s4ayGSZIj9hEeMVH+2G7",
    "nQ8I2xw/j5JZ9KvfUpeY4RYNGgECA2PWvtgcDOge+zrNQ0T9qHNoCB3uFlBh6ttaIKhqLjnzERc4uYo5LDUR1rBP+bEI6GKK+7Lr",
    "gE2jiG+PHj17dnqKkvfED6um6sqGXLkIddvdfeVYHsMtyYfdBa7ienR69PqJzqhmK5RRaDPAF0Ewh8M6+xGcJNEviOnlzs7ZK+Mp",
    "X8muzN1tcYZYUUAUmrZE3Tc2ldcAs+o+oLjLlupau/e49/TAWon9wzWI+r2nB4977V6jXKC7+/RRu7tS4CLBHW1UoN19+qi72+AO",
    "o3MpSRvA6WWNRQZ1U9XupunKpohEpvOo8J1eEgVBK5AX/rmPuyAx5Wxn3Ge/cWMD3s9NwCtRoFSUDAJCTWvyIPBNeombZlQ4d0JA",
    "yOU5I2XZgWilkfdyZzlc1eQGqfjLrjabIKFORbOpKclFzqJffA2HDzXbBoSS/ZI5jLQ5vHJTTORiJ4yEKlOr8akfpqd86teHuFhp",
    "y4sTLVvyVXDK/MD4QlBzWLoUdVVdGIWtUF5EmQ/ntEYOm2+zdBiAk0lemRd6KGr6+BoN0WFW+AiPxhlx6XAwmrv/9zkoxWoALoSK",
    "045vRRyBenugorstm6IzmCTRrzJs2CjfatsPsZxoHc6AsIlUk+dEoZPST0kzkpnG5eJV6RpAwodDPfx5ygF0WB+8wo/mmLJsDn2P",
    "urgrHKFY4mo5OBS28tDZGd8KKhCgPo9XXyNS/NT1OchCXMR0Y/uXeYDAa+WLD7HE2FmI76y3xVPWEZzLCS59y00CcGpOFBVUPpAT",
    "9sCDXd07ouMI/0JNU7dcKp1NU11rzwfJCQ0j0YXHhOMtIskIOcYRkTgGGvyeaHma69dpF1FWO8qSh68PdEUKBRLN+lHqxlOL09LU",
    "nVSTJkyFWA1D6YomvcUbPvpl+hUr+oVsm+gaF0gvoyhAjjP+gaQL9qoksuzxrXHP+PWEX7n+iG+4IowNVQvQcUp4ygmYEnNbpQ51",
    "AtKiLGta2phn5QoCNU2Va2VXb6Xlq/lgBlRz86swv0dowL3W7p5YSjcRkbp9k62S1IW3KvoyX6eHeRj7CYe4QKw7TPpMjlvyCht5",
    "wDsSKAQCIYJyTjKEgdclgyUh7RLaPiAHtlncHnHCdbiB2kKmy5h2+zO+QFArhcPShbs6uhVfk0holJq7GVf7aW+Fke2zfbbP9tk+",
    "22f7bJ/ts322z/bZPttn+2yf7bN9ts/22T7bZ/tsn+2zfbbP9tk+22f7bJ/ts33Wn/8Ll9dXTQDwAAA=",
]
blob = base64.b64decode("".join(BLOB_PARTS))
tarfile.open(fileobj=io.BytesIO(blob), mode="r:gz").extractall(".")
print("\u2705 Engine loaded.")

### 3️⃣ Your settings — edit the two lines, then click ▶

- **VIDEO_URL** — paste a YouTube link, e.g. a Flock Talk episode
- **ANTHROPIC_API_KEY** — get one at [console.anthropic.com](https://console.anthropic.com) → API Keys → Create Key (starts with `sk-ant-`)

In [ ]:
VIDEO_URL = "https://youtu.be/PASTE_YOUR_VIDEO_LINK"  #@param {type:"string"}
ANTHROPIC_API_KEY = "sk-ant-PASTE_YOUR_KEY"           #@param {type:"string"}
MAX_CLIPS = 3                                          #@param {type:"integer"}
print("✅ Saved. Video:", VIDEO_URL, "| clips:", MAX_CLIPS, "— go to step 4.")

### 4️⃣ Make the clips. Click ▶ and let it cook (5–15 min; the first run also downloads the speech model).

In [ ]:
import os
os.environ["ANTHROPIC_API_KEY"] = ANTHROPIC_API_KEY
os.environ["CLIP_ENGINE_ASR"] = "faster_whisper"

from pathlib import Path
from clip_engine.render import process

clips = process(source=VIDEO_URL, out_dir=Path("OUT"), work_root=Path("work"),
                mode="talk", max_clips=int(MAX_CLIPS))
print(f"\n✅ Done — {len(clips)} clips made. Go to step 5 to download.")
for c in clips: print("   •", c.name)

### 5️⃣ Download your clips. Click ▶ — `clips.zip` saves to your computer.

In [ ]:
!zip -qr clips.zip OUT
from google.colab import files
files.download("clips.zip")
print("✅ If no download started: click the 📁 folder icon on the left, right-click clips.zip → Download.")

---
**Tweak the look:** open the 📁 folder icon on the left → `config` → double-click `brand.yaml` — fonts, highlight colors, clip length all live there. Re-run step 4 after editing.

**Something errored?** Copy the red text and paste it to Claude — it built this and will fix it.